# Time series

Annual global time series from the HAMOCC annual history files (`hbgcy`): volume-weighted mean concentrations (full depth and 0-100 m) and area-integrated fluxes.

In [ ]:
import matplotlib.pyplot as plt
import yaml
from pyprojroot import here

from hamocc import io, plots, timeseries, variables

In [ ]:
# Optional second case for comparison. year_offset aligns its model years with the test case.
caseid2 = None
year_start2 = None
year_end2 = None
year_offset2 = 0

In [ ]:
with open(here('notebooks_hamocc/settings.yml')) as f:
    cfg = yaml.safe_load(f)
exp = cfg['experiment']
grid_file = cfg['grids'][exp['grid']]['grid_file']

def load_case(caseid, ys, ye):
    files = io.history_files(exp['input_root'], caseid, 'hbgcy', ys, ye)
    return io.open_history(files, cfg['variables']['hbgcy'])

cases = [(exp['caseid'], load_case(exp['caseid'], exp['year_start'], exp['year_end']), 0)]
if caseid2:
    cases.append((caseid2, load_case(caseid2, year_start2, year_end2), year_offset2))

parea = timeseries.load_parea(grid_file)
labels = [c[0] for c in cases]

## Global mean concentrations

Volume-weighted over the full ocean (weights: z-level thickness from `depth_bnds` times cell area, using the z-level fields `*lvl`).

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 9), sharex=True)
for ax, name in zip(axes.flat, variables.CONC_TS_VARS):
    series = [timeseries.global_mean_series(ds, name, parea, offset=off).load() for _, ds, off in cases]
    plots.plot_series(series, labels, ax=ax)
fig.tight_layout()

## Upper-ocean (0-100 m) mean concentrations

From the z-level fields (`*lvl`), layers clipped to 0-100 m.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 9), sharex=True)
for ax, name in zip(axes.flat, variables.CONC_TS_VARS):
    series = [timeseries.global_mean_series(ds, name, parea, zmax=100.0, offset=off).load() for _, ds, off in cases]
    plots.plot_series(series, labels, ax=ax, title=f"{series[0].attrs['long_name']}, 0-100 m")
fig.tight_layout()

## Area-integrated fluxes

Annual totals over the global ocean. The net CO2 flux is positive into the ocean (`co2fxd - co2fxu`).

In [ ]:
flux_names = ['co2fxn', 'ppint', 'epc100', 'epcalc100', 'dmsflux']
fig, axes = plt.subplots(3, 2, figsize=(12, 9), sharex=True)
for ax, name in zip(axes.flat, flux_names):
    series = [timeseries.flux_series(ds, name, parea, offset=off).load() for _, ds, off in cases]
    plots.plot_series(series, labels, ax=ax)
fig.tight_layout()